In [2]:
import os
import glob
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Load Dataset**

In [3]:
import pandas as pd
import numpy as np
import time

In [4]:
train_dir = "/content/drive/MyDrive/DeepFake Detection Project/Datasets/Batch_size_8/Train/"
validation_dir = "/content/drive/MyDrive/DeepFake Detection Project/Datasets/Batch_size_8/Validation/"
test_dir = "/content/drive/MyDrive/DeepFake Detection Project/Datasets/Batch_size_8/Test/"

In [5]:
training_results = pd.read_excel("/content/drive/MyDrive/DeepFake Detection Project/Results/3PI_TDCNN_ConvLSTM/Train_Metrics.xlsx")
validation_results = pd.read_excel("/content/drive/MyDrive/DeepFake Detection Project/Results/3PI_TDCNN_ConvLSTM/Validation_Metrics.xlsx")

In [8]:
import tensorflow as tf
from tensorflow import keras
from keras.models import Model
from keras.layers import (
    Input, Conv2D, MaxPooling2D, MaxPooling1D, Flatten, LSTM, ConvLSTM2D , Dense, TimeDistributed, Dropout,
    GlobalAveragePooling2D, Concatenate, AveragePooling2D, BatchNormalization, Average,
    Reshape, AveragePooling3D, MaxPooling3D, GlobalMaxPooling3D
)

# **Define Model Architecture**

In [9]:
def build_3PTCSL(input_shape=(10, 256, 256, 3), hidden_dim=64, num_classes=1):

    in1 = Input(shape=input_shape, name="Input_Layer_1")
    in2 = Input(shape=input_shape, name="Input_Layer_2")
    in3 = Input(shape=input_shape, name="Input_Layer_3")

    cnn = tf.keras.Sequential([
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        MaxPooling2D(pool_size=(2, 2)),
    ], name="CNN_Module")

    x1 = TimeDistributed(cnn, name="Time_Distributed_CNN_1")(in1)
    x2 = TimeDistributed(cnn, name="Time_Distributed_CNN_2")(in2)
    x3 = TimeDistributed(cnn, name="Time_Distributed_CNN_3")(in3)

    x = Average(name="Averaging_Layer")([x1, x2, x3])

    x = ConvLSTM2D(hidden_dim, (3, 3), padding='same', return_sequences=True, name="ConvLSTM_Layer", activation='tanh')(x)
    x = GlobalMaxPooling3D()(x)
    x = Flatten(name="Flatten_Layer")(x)
    x = Dense(128, activation='relu', name="Dense_Layer_1")(x)
    x = Dropout(0.5, name="Dropout_Layer")(x)
    x = Dense(num_classes, activation='sigmoid', name="Output_Layer")(x)
    model = Model(inputs=[in1, in2, in3], outputs=x, name="3PTCSL")
    model.summary()
    return model

# **Training Loop**

In [ ]:
# Build & Compile Model Once
model = build_3PTCSL(input_shape=(10,256,256,3))
model.save_weights("/content/drive/MyDrive/DeepFake Detection Project/Results/3PI_TDCNN_ConvLSTM/model.weights.h5")
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy','AUC','precision','recall']
)

Model: "3PTCSL"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Input_Layer_1       │ (None, 10, 256,   │          0 │ -                 │
│ (InputLayer)        │ 256, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Input_Layer_2       │ (None, 10, 256,   │          0 │ -                 │
│ (InputLayer)        │ 256, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Input_Layer_3       │ (None, 10, 256,   │          0 │ -                 │
│ (InputLayer)        │ 256, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Time_Distributed_C… │ (None, 10, 32,    │     93,248 │ Input_Layer_1[0]… │
│ (TimeDistributed)   │ 32, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Time_Distributed_C… │ (None, 10, 32,    │     93,248 │ Input_Layer_2[0]… │
│ (TimeDistributed)   │ 32, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Time_Distributed_C… │ (None, 10, 32,    │     93,248 │ Input_Layer_3[0]… │
│ (TimeDistributed)   │ 32, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Averaging_Layer     │ (None, 10, 32,    │          0 │ Time_Distributed… │
│ (Average)           │ 32, 128)          │            │ Time_Distributed… │
│                     │                   │            │ Time_Distributed… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ConvLSTM_Layer      │ (None, 10, 32,    │    442,624 │ Averaging_Layer[… │
│ (ConvLSTM2D)        │ 32, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 64)        │          0 │ ConvLSTM_Layer[0… │
│ (GlobalMaxPooling3… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Flatten_Layer       │ (None, 64)        │          0 │ global_max_pooli… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dense_Layer_1       │ (None, 128)       │      8,320 │ Flatten_Layer[0]… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dropout_Layer       │ (None, 128)       │          0 │ Dense_Layer_1[0]… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Output_Layer        │ (None, 1)         │        129 │ Dropout_Layer[0]… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 544,321 (2.08 MB)

 Trainable params: 544,321 (2.08 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Early‐Stopping Setup
best_val_loss = np.inf      # best validation loss seen so far
patience = 8                # how many epochs to wait after no improvement
wait = 0                    # counter for epochs with no improvement


for i in range(0,100):
    model = build_3PTCSL(input_shape=(10,256,256,3))
    model.load_weights(f"/content/drive/MyDrive/DeepFake Detection Project/Results/3PI_TDCNN_ConvLSTM/model.weights.h5")
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy','AUC','precision','recall']
    )
    epoch_loss = []
    epoch_accuracy = []
    epoch_auc = []
    epoch_precision = []
    epoch_recall = []
    start_time = time.time()

    # Training Loop
    for j in range(0, 105):
        X0 = np.load(train_dir+f"Batch{j}X0.npy")
        print(f"Mounted Batch{j}X0")
        X1 = np.load(train_dir+f"Batch{j}X1.npy")
        print(f"Mounted Batch{j}X1")
        X2 = np.load(train_dir+f"Batch{j}X2.npy")
        print(f"Mounted Batch{j}X2")
        y = np.load(train_dir+f"Batch{j}y.npy")
        print(f"Mounted Batch{j}y")
        print(f"Epoch {i} of {100} ::: Training Batch {j+1} of {105}")
        history = model.fit([X0, X1, X2], y, epochs=1, verbose=0)

        # Collect training metrics
        epoch_loss.append(history.history['loss'][0])
        epoch_accuracy.append(history.history['accuracy'][0])
        epoch_auc.append(history.history['AUC'][0])
        epoch_precision.append(history.history['precision'][0])
        epoch_recall.append(history.history['recall'][0])
        print(f"Batch {i+1} Loss: {history.history['loss'][0]}; Accuracy: {history.history['accuracy'][0]}; AUC: {history.history['AUC'][0]}; Precision: {history.history['precision'][0]}; Recall: {history.history['recall'][0]} \n")

    # Validation Loop
    val_epoch_loss = []
    val_epoch_accuracy = []
    val_epoch_auc = []
    val_epoch_precision = []
    val_epoch_recall = []

    for j in range(0, 13):
        val_X0 = np.load(validation_dir+f"Batch{j}X0.npy")
        val_X1 = np.load(validation_dir+f"Batch{j}X1.npy")
        val_X2 = np.load(validation_dir+f"Batch{j}X2.npy")
        val_y = np.load(validation_dir+f"Batch{j}y.npy")
        print(f"Epoch {i} of {100} ::: Validation Batch {j+1} of 5")
        val_metrics = model.evaluate([val_X0, val_X1, val_X2], val_y, verbose=0)

        # Collect validation metrics
        val_epoch_loss.append(val_metrics[0])
        val_epoch_accuracy.append(val_metrics[1])
        val_epoch_auc.append(val_metrics[2])
        val_epoch_precision.append(val_metrics[3])
        val_epoch_recall.append(val_metrics[4])

    end_time = time.time()
    elapsed_time = end_time - start_time

    # Average the training metrics for the epoch
    loss_per_epoch = np.mean(epoch_loss)
    accuracy_per_epoch = np.mean(epoch_accuracy)
    auc_per_epoch = np.mean(epoch_auc)
    precision_per_epoch = np.mean(epoch_precision)
    recall_per_epoch = np.mean(epoch_recall)

    metrics = {
        'Loss': loss_per_epoch,
        'Accuracy': accuracy_per_epoch,
        'AUC': auc_per_epoch,
        'Precision': precision_per_epoch,
        'Recall': recall_per_epoch
    }
    training_results = pd.concat([training_results, pd.DataFrame([metrics])], ignore_index=True)
    training_results.to_excel("/content/drive/MyDrive/DeepFake Detection Project/Results/3PI_TDCNN_ConvLSTM/Train_Metrics.xlsx", index=False)

    # Average the validation metrics for the epoch
    val_loss_per_epoch = np.mean(val_epoch_loss)
    val_accuracy_per_epoch = np.mean(val_epoch_accuracy)
    val_auc_per_epoch = np.mean(val_epoch_auc)
    val_precision_per_epoch = np.mean(val_epoch_precision)
    val_recall_per_epoch = np.mean(val_epoch_recall)

    metrics = {
        'Loss': val_loss_per_epoch,
        'Accuracy': val_accuracy_per_epoch,
        'AUC': val_auc_per_epoch,
        'Precision': val_precision_per_epoch,
        'Recall': val_recall_per_epoch
    }
    validation_results = pd.concat([validation_results, pd.DataFrame([metrics])], ignore_index=True)
    validation_results.to_excel("/content/drive/MyDrive/DeepFake Detection Project/Results/3PI_TDCNN_ConvLSTM/Validation_Metrics.xlsx", index=False)

    # Print metrics for the epoch
    print(f"\nEpoch {i} completed in {elapsed_time:.2f} seconds")
    print(f"Training - Loss: {loss_per_epoch:.4f}, Accuracy: {accuracy_per_epoch:.4f}, "
          f"AUC: {auc_per_epoch:.4f}, Precision: {precision_per_epoch:.4f}, Recall: {recall_per_epoch:.4f}")
    print(f"Validation - Loss: {val_loss_per_epoch:.4f}, Accuracy: {val_accuracy_per_epoch:.4f}, "
          f"AUC: {val_auc_per_epoch:.4f}, Precision: {val_precision_per_epoch:.4f}, Recall: {val_recall_per_epoch:.4f}")

    # Early Stopping Check
    if val_loss_per_epoch < best_val_loss - 1e-3:
        best_val_loss = val_loss_per_epoch
        wait = 0
        # save best weights
        model.save_weights(
            "/content/drive/MyDrive/DeepFake Detection Project/Results/3PI_TDCNN_ConvLSTM/best_model.weights.h5"
        )
    else:
        wait += 1
        if wait >= patience:
            print(f"\nEarly stopping at epoch {i+1}. "
                  f"Restoring best model from epoch {i+1-wait}.")
            model.load_weights(
                "/content/drive/MyDrive/DeepFake Detection Project/Results/3PI_TDCNN_ConvLSTM/best_model.weights.h5"
            )
            break

    # Save checkpoint every epochs
    model.save_weights(f"/content/drive/MyDrive/DeepFake Detection Project/Results/3PI_TDCNN_ConvLSTM/model.weights.h5")

Model: "3PTCSL"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Input_Layer_1       │ (None, 10, 256,   │          0 │ -                 │
│ (InputLayer)        │ 256, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Input_Layer_2       │ (None, 10, 256,   │          0 │ -                 │
│ (InputLayer)        │ 256, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Input_Layer_3       │ (None, 10, 256,   │          0 │ -                 │
│ (InputLayer)        │ 256, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Time_Distributed_C… │ (None, 10, 32,    │     93,248 │ Input_Layer_1[0]… │
│ (TimeDistributed)   │ 32, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Time_Distributed_C… │ (None, 10, 32,    │     93,248 │ Input_Layer_2[0]… │
│ (TimeDistributed)   │ 32, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Time_Distributed_C… │ (None, 10, 32,    │     93,248 │ Input_Layer_3[0]… │
│ (TimeDistributed)   │ 32, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Averaging_Layer     │ (None, 10, 32,    │          0 │ Time_Distributed… │
│ (Average)           │ 32, 128)          │            │ Time_Distributed… │
│                     │                   │            │ Time_Distributed… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ConvLSTM_Layer      │ (None, 10, 32,    │    442,624 │ Averaging_Layer[… │
│ (ConvLSTM2D)        │ 32, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 64)        │          0 │ ConvLSTM_Layer[0… │
│ (GlobalMaxPooling3… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Flatten_Layer       │ (None, 64)        │          0 │ global_max_pooli… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dense_Layer_1       │ (None, 128)       │      8,320 │ Flatten_Layer[0]… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dropout_Layer       │ (None, 128)       │          0 │ Dense_Layer_1[0]… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Output_Layer        │ (None, 1)         │        129 │ Dropout_Layer[0]… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 544,321 (2.08 MB)

 Trainable params: 544,321 (2.08 MB)

 Non-trainable params: 0 (0.00 B)

Mounted Batch0X0
Mounted Batch0X1
Mounted Batch0X2
Mounted Batch0y
Epoch 0 of 100 ::: Training Batch 1 of 105
Batch 1 Loss: 0.6904280185699463; Accuracy: 0.5; AUC: 0.5; Precision: 0.0; Recall: 0.0 

Mounted Batch1X0
Mounted Batch1X1
Mounted Batch1X2
Mounted Batch1y
Epoch 0 of 100 ::: Training Batch 2 of 105
Batch 1 Loss: 0.656177818775177; Accuracy: 0.75; AUC: 0.4583333134651184; Precision: 0.0; Recall: 0.0 

Mounted Batch2X0
Mounted Batch2X1
Mounted Batch2X2
Mounted Batch2y
Epoch 0 of 100 ::: Training Batch 3 of 105
Batch 1 Loss: 0.7209128141403198; Accuracy: 0.625; AUC: 0.2666666507720947; Precision: 0.0; Recall: 0.0 

Mounted Batch3X0
Mounted Batch3X1
Mounted Batch3X2
Mounted Batch3y
Epoch 0 of 100 ::: Training Batch 4 of 105
Batch 1 Loss: 0.6797221899032593; Accuracy: 0.625; AUC: 0.46666666865348816; Precision: 0.0; Recall: 0.0 

Mounted Batch4X0
Mounted Batch4X1
Mounted Batch4X2
Mounted Batch4y
Epoch 0 of 100 ::: Training Batch 5 of 105
Batch 1 Loss: 0.8545728921890259; Accuracy: 

... (70 outputs left out)

 Total params: 544,321 (2.08 MB)

 Trainable params: 544,321 (2.08 MB)

 Non-trainable params: 0 (0.00 B)

Mounted Batch0X0
Mounted Batch0X1
Mounted Batch0X2
Mounted Batch0y
Epoch 12 of 100 ::: Training Batch 1 of 105
Batch 13 Loss: 0.47522398829460144; Accuracy: 0.75; AUC: 0.8333333134651184; Precision: 0.5; Recall: 1.0 

Mounted Batch1X0
Mounted Batch1X1
Mounted Batch1X2
Mounted Batch1y
Epoch 12 of 100 ::: Training Batch 2 of 105
Batch 13 Loss: 0.5385689735412598; Accuracy: 0.625; AUC: 0.5833333134651184; Precision: 0.0; Recall: 0.0 

Mounted Batch2X0
Mounted Batch2X1
Mounted Batch2X2
Mounted Batch2y
Epoch 12 of 100 ::: Training Batch 3 of 105
Batch 13 Loss: 0.6362271904945374; Accuracy: 0.625; AUC: 0.7333333492279053; Precision: 0.5; Recall: 0.6666666865348816 

Mounted Batch3X0
Mounted Batch3X1
Mounted Batch3X2
Mounted Batch3y
Epoch 12 of 100 ::: Training Batch 4 of 105
Batch 13 Loss: 0.4418848156929016; Accuracy: 0.75; AUC: 0.8666666746139526; Precision: 1.0; Recall: 0.3333333432674408 

Mounted Batch4X0
Mounted Batch4X1
Mounted Batch4X2
Mounted Batch4y
Epoch 12 of 100 ::: Training Bat

# **Testing**

In [10]:
model = build_3PTCSL(input_shape=(10,256,256,3))
model.load_weights(f"/content/drive/MyDrive/DeepFake Detection Project/Results/3PI_TDCNN_ConvLSTM/model.weights.h5")
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy','AUC','precision','recall'])

Model: "3PTCSL"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Input_Layer_1       │ (None, 10, 256,   │          0 │ -                 │
│ (InputLayer)        │ 256, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Input_Layer_2       │ (None, 10, 256,   │          0 │ -                 │
│ (InputLayer)        │ 256, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Input_Layer_3       │ (None, 10, 256,   │          0 │ -                 │
│ (InputLayer)        │ 256, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Time_Distributed_C… │ (None, 10, 32,    │     93,248 │ Input_Layer_1[0]… │
│ (TimeDistributed)   │ 32, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Time_Distributed_C… │ (None, 10, 32,    │     93,248 │ Input_Layer_2[0]… │
│ (TimeDistributed)   │ 32, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Time_Distributed_C… │ (None, 10, 32,    │     93,248 │ Input_Layer_3[0]… │
│ (TimeDistributed)   │ 32, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Averaging_Layer     │ (None, 10, 32,    │          0 │ Time_Distributed… │
│ (Average)           │ 32, 128)          │            │ Time_Distributed… │
│                     │                   │            │ Time_Distributed… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ConvLSTM_Layer      │ (None, 10, 32,    │    442,624 │ Averaging_Layer[… │
│ (ConvLSTM2D)        │ 32, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 64)        │          0 │ ConvLSTM_Layer[0… │
│ (GlobalMaxPooling3… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Flatten_Layer       │ (None, 64)        │          0 │ global_max_pooli… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dense_Layer_1       │ (None, 128)       │      8,320 │ Flatten_Layer[0]… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dropout_Layer       │ (None, 128)       │          0 │ Dense_Layer_1[0]… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Output_Layer        │ (None, 1)         │        129 │ Dropout_Layer[0]… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 544,321 (2.08 MB)

 Trainable params: 544,321 (2.08 MB)

 Non-trainable params: 0 (0.00 B)

In [23]:
loss, accuracy, precision, recall, auc = list(), list(), list(), list(), list()
for i in range(0,6):
    if i!=4 and i!=2:
      test_X0 = np.load(test_dir+f"Batch{i}X0.npy")
      test_X1 = np.load(test_dir+f"Batch{i}X1.npy")
      test_X2 = np.load(test_dir+f"Batch{i}X2.npy")
      test_y = np.load(test_dir+f"Batch{i}y.npy")
      print(f"Test Batch {i} of 6")
      test_metrics = model.evaluate([test_X0, test_X1, test_X2], test_y)
      loss.append(test_metrics[0])
      accuracy.append(test_metrics[1])
      precision.append(test_metrics[2])
      recall.append(test_metrics[3])
      auc.append(test_metrics[4])
for i in range(0,12):
    if i==6 or i==9:
        test_X0 = np.load(validation_dir+f"Batch{i}X0.npy")
        test_X1 = np.load(validation_dir+f"Batch{i}X1.npy")
        test_X2 = np.load(validation_dir+f"Batch{i}X2.npy")
        test_y = np.load(validation_dir+f"Batch{i}y.npy")
        print(f"Validation Batch {i} of 5")
        test_metrics = model.evaluate([test_X0, test_X1, test_X2], test_y)
        loss.append(test_metrics[0])
        accuracy.append(test_metrics[1])
        precision.append(test_metrics[2])
        recall.append(test_metrics[3])
        auc.append(test_metrics[4])


print(f"Loss::{np.mean(loss)}")
print(f"Accuracy::{np.mean(accuracy)}")
print(f"Precision::{np.mean(precision)}")
print(f"Recall::{np.mean(recall)}")
print(f"AUC::{np.mean(auc)}")

Test Batch 0 of 6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 337ms/step - AUC: 1.0000 - accuracy: 0.8750 - loss: 0.3963 - precision: 1.0000 - recall: 0.8333
Test Batch 1 of 6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 288ms/step - AUC: 1.0000 - accuracy: 1.0000 - loss: 0.2723 - precision: 1.0000 - recall: 1.0000
Test Batch 3 of 6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 266ms/step - AUC: 1.0000 - accuracy: 1.0000 - loss: 0.3855 - precision: 1.0000 - recall: 1.0000
Test Batch 5 of 6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 266ms/step - AUC: 0.8667 - accuracy: 0.8750 - loss: 0.3579 - precision: 0.8333 - recall: 1.0000
Validation Batch 6 of 5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 281ms/step - AUC: 1.0000 - accuracy: 1.0000 - loss: 0.3481 - precision: 1.0000 - recall: 1.0000
Validation Batch 9 of 5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 351ms/step - AUC: 1.0000 - accuracy: 0.8750 - loss: 0.4176 - precision: 0.8333 - recall: 1.0000
Loss::0.36294025182724
Accuracy::0.9375
Precision::0.9777777791023254
Recall::0.9444444378217062
AUC::0.972222218910853


In [25]:
f1 = 2*(0.978*0.944)/(0.978+0.944)
f1

0.9606992715920916